# Lab 07 — Reed-Solomon from Scratch

Labs 04 and 06 showed *what* RustFS does when drives fail — it reconstructs data transparently.
This lab shows *how*. We implement **RS(4,2)** in pure Python (~150 lines, no libraries)
and verify it produces the same results as the server.

### What you will build

1. **GF(2⁸)** — arithmetic in a Galois Field where addition = XOR and multiplication uses log tables
2. **Vandermonde matrix** — the generator matrix that produces parity shards from data shards
3. **Encoder** — split a byte string into k=4 data shards, compute m=2 parity shards
4. **Decoder** — reconstruct the original data from any k=4 surviving shards out of 6

### Why this matters

RustFS uses hardware-accelerated RS (AVX2 SIMD, Blake3 checksums) but the *math* is identical.
Understanding it lets you reason about:
- Why k=4 is the minimum — not k=3, not k=5
- Why adding more parity shards costs storage but increases fault tolerance
- Why RS is better than 3× replication for large objects


---
## 1 · Galois Field GF(2⁸)

Standard integers don't work for error correction because division produces fractions.
GF(2⁸) is a 256-element field where:
- **Addition** = XOR (no carries, always stays in range)
- **Multiplication** = polynomial multiplication modulo the irreducible polynomial x⁸+x⁴+x³+x²+1

We pre-compute log/antilog tables so every multiply is O(1).

In [ ]:
# Galois Field GF(2^8) with primitive polynomial x^8+x^4+x^3+x^2+1 (0x11d)
GF_POLY = 0x11d
GF_SIZE = 256

# Build exp (antilog) and log tables
_exp = [0] * (GF_SIZE * 2)
_log = [0] * GF_SIZE

x = 1
for i in range(GF_SIZE - 1):
    _exp[i] = x
    _log[x] = i
    x <<= 1
    if x >= GF_SIZE:
        x ^= GF_POLY

for i in range(GF_SIZE - 1, GF_SIZE * 2):
    _exp[i] = _exp[i - (GF_SIZE - 1)]


def gf_mul(a: int, b: int) -> int:
    if a == 0 or b == 0:
        return 0
    return _exp[_log[a] + _log[b]]


def gf_div(a: int, b: int) -> int:
    if b == 0:
        raise ZeroDivisionError
    if a == 0:
        return 0
    return _exp[_log[a] - _log[b] + (GF_SIZE - 1)]


def gf_pow(x: int, power: int) -> int:
    return _exp[(_log[x] * power) % (GF_SIZE - 1)]


# Sanity checks
assert gf_mul(3, 7) == gf_mul(7, 3),          'multiplication must be commutative'
assert gf_mul(0, 99) == 0,                     'zero property'
assert gf_div(gf_mul(5, 11), 11) == 5,        'div undoes mul'
assert 5 ^ 5 == 0,                             'XOR addition: a + a = 0 in GF(2^n)'

print('✅ GF(2^8) tables built — 255 non-zero elements, addition=XOR')
print(f'   Example: gf_mul(2, 3) = {gf_mul(2, 3)}  (not 6 — no carry in GF!)')
print(f'   Example: 5 XOR 5      = {5 ^ 5}    (every element is its own additive inverse)')

---
## 2 · Matrix Operations over GF(2⁸)

The encoder multiplies a **generator matrix** (k+m × k) by a data vector.
All arithmetic is in GF(2⁸) — no ordinary +, ×, or /.

In [ ]:
Matrix = list[list[int]]  # type alias


def mat_mul(A: Matrix, B: Matrix) -> Matrix:
    rows_a, cols_a = len(A), len(A[0])
    rows_b, cols_b = len(B), len(B[0])
    assert cols_a == rows_b
    result = [[0] * cols_b for _ in range(rows_a)]
    for i in range(rows_a):
        for j in range(cols_b):
            acc = 0
            for k in range(cols_a):
                acc ^= gf_mul(A[i][k], B[k][j])
            result[i][j] = acc
    return result


def mat_invert(M: Matrix) -> Matrix:
    """Invert a square matrix over GF(2^8) using Gaussian elimination."""
    n = len(M)
    # Augment with identity
    aug = [row[:] + ([1 if i == j else 0 for j in range(n)]) for i, row in enumerate(M)]
    for col in range(n):
        # Find pivot
        pivot = next((r for r in range(col, n) if aug[r][col] != 0), None)
        if pivot is None:
            raise ValueError('Matrix is singular — not invertible')
        aug[col], aug[pivot] = aug[pivot], aug[col]
        # Scale pivot row
        scale = gf_div(1, aug[col][col])
        aug[col] = [gf_mul(v, scale) for v in aug[col]]
        # Eliminate column
        for row in range(n):
            if row != col and aug[row][col] != 0:
                factor = aug[row][col]
                aug[row] = [aug[row][j] ^ gf_mul(factor, aug[col][j]) for j in range(2 * n)]
    return [row[n:] for row in aug]


print('✅ Matrix operations over GF(2^8) ready')

---
## 3 · Vandermonde Generator Matrix

The Vandermonde matrix has a special property: **any k×k submatrix is invertible**.
This is what guarantees that any k surviving shards can reconstruct the original data.

For RS(k=4, m=2) the generator is a 6×4 matrix:
- Top 4 rows = identity (data shards pass through unchanged)
- Bottom 2 rows = Vandermonde entries (the parity shards)

In [ ]:
def build_generator(k: int, m: int) -> Matrix:
    """Build a systematic Vandermonde generator matrix of shape (k+m) x k.
    Top k rows are the identity; bottom m rows are Vandermonde parity."""
    # Identity block
    G = [[1 if r == c else 0 for c in range(k)] for r in range(k)]
    # Vandermonde parity rows: row i = [alpha^(i*0), alpha^(i*1), ..., alpha^(i*(k-1))]
    for i in range(1, m + 1):
        G.append([gf_pow(i, j) for j in range(k)])
    return G


K, M = 4, 2  # RS(4,2): 4 data shards + 2 parity shards
N = K + M    # 6 total shards

G = build_generator(K, M)

print(f'Generator matrix G for RS({K},{M}) — shape {N}×{K}:')
print(f'  (rows 0-{K-1}: identity = data shards pass through unchanged)')
print(f'  (rows {K}-{N-1}: Vandermonde = parity shards)')
print()
for i, row in enumerate(G):
    label = f'  shard {i} (data   )' if i < K else f'  shard {i} (parity )'
    print(f'{label}: [{" ".join(f"{v:3d}" for v in row)}]')

---
## 4 · Encoding

Split input data into k=4 equal shards, then multiply by the generator matrix
to produce m=2 parity shards. The final 6-shard set can survive any 2 shard losses.

In [ ]:
def encode(data: bytes, k: int, m: int) -> list[bytes]:
    """Encode data into k+m shards using RS(k,m)."""
    # Pad to a multiple of k
    pad = (-len(data)) % k
    data = data + bytes(pad)
    shard_size = len(data) // k

    # Split into k data shards
    data_shards = [data[i * shard_size:(i + 1) * shard_size] for i in range(k)]

    G = build_generator(k, m)
    parity_shards = [bytearray(shard_size) for _ in range(m)]

    # For each byte position, multiply the column vector by the parity rows of G
    for pos in range(shard_size):
        for pi in range(m):
            val = 0
            for di in range(k):
                val ^= gf_mul(G[k + pi][di], data_shards[di][pos])
            parity_shards[pi][pos] = val

    return data_shards + [bytes(p) for p in parity_shards]


# Test with a small message
MESSAGE = b'Hello, Reed-Solomon! This message will survive two drive failures.'

shards = encode(MESSAGE, K, M)

print(f'Original message: {MESSAGE!r}')
print(f'Message length:   {len(MESSAGE)} bytes')
print()
print(f'Encoded into {N} shards of {len(shards[0])} bytes each:')
for i, s in enumerate(shards):
    kind = 'data  ' if i < K else 'parity'
    print(f'  shard {i} ({kind}): {s.hex()}')

storage_overhead = N / K
print(f'\nStorage overhead: {N}/{K} = {storage_overhead:.2f}x  '
      f'(vs {K*2}/{K} = {K*2/K:.2f}x for 3× replication)')

---
## 5 · Decoding — Reconstruction from Any 4 Shards

Given k=4 surviving shards (indices 0–5, any combination), we:
1. Extract the k rows of G corresponding to the available shards → submatrix
2. Invert the submatrix over GF(2⁸)
3. Multiply the inverted matrix by the surviving shards to recover the original data

In [ ]:
def decode(available_shards: dict[int, bytes], k: int, m: int, original_len: int) -> bytes:
    """Reconstruct original data from any k surviving shards.
    available_shards: dict mapping shard index → shard bytes.
    """
    assert len(available_shards) >= k, f'Need at least {k} shards, got {len(available_shards)}'

    G = build_generator(k, m)
    indices = sorted(available_shards)[:k]  # pick exactly k

    # Submatrix: rows of G for the chosen shard indices
    sub_G = [G[i] for i in indices]
    inv_G = mat_invert(sub_G)

    shard_size = len(next(iter(available_shards.values())))
    result = bytearray(k * shard_size)

    for pos in range(shard_size):
        col = [available_shards[i][pos] for i in indices]
        for di in range(k):
            val = 0
            for j in range(k):
                val ^= gf_mul(inv_G[di][j], col[j])
            result[di * shard_size + pos] = val

    return bytes(result[:original_len])


# --- Scenario 1: no failures ---
recovered = decode({i: shards[i] for i in range(N)}, K, M, len(MESSAGE))
assert recovered == MESSAGE
print('✅ Scenario 1 (all 6 shards): recovered correctly')

# --- Scenario 2: lose shards 4 and 5 (both parity shards) ---
available = {i: shards[i] for i in [0, 1, 2, 3]}
recovered = decode(available, K, M, len(MESSAGE))
assert recovered == MESSAGE
print('✅ Scenario 2 (lost parity 4,5): recovered from 4 data shards')

# --- Scenario 3: lose shards 0 and 1 (two data shards) ---
available = {i: shards[i] for i in [2, 3, 4, 5]}
recovered = decode(available, K, M, len(MESSAGE))
assert recovered == MESSAGE
print('✅ Scenario 3 (lost data 0,1): recovered from shards 2,3 + parity 4,5')

# --- Scenario 4: lose one data + one parity ---
available = {i: shards[i] for i in [0, 2, 3, 5]}
recovered = decode(available, K, M, len(MESSAGE))
assert recovered == MESSAGE
print('✅ Scenario 4 (lost data 1, parity 4): recovered from mixed surviving shards')

# --- Scenario 5: lose 3 shards — should fail ---
available = {i: shards[i] for i in [0, 1, 2]}
try:
    decode(available, K, M, len(MESSAGE))
    print('❌ Scenario 5: should have failed!')
except (AssertionError, ValueError):
    print('✅ Scenario 5 (lost 3 shards): correctly raises error — unrecoverable')

---
## 6 · Verify Against RustFS

Upload an object to RustFS, simulate a drive failure, read back, and confirm
the server reconstructs the same bytes our Python decoder would recover.

In [ ]:
import sys
sys.path.insert(0, '../scripts')
from lab_utils import get_s3_client, ensure_bucket, fail_drive, restore_drive, cleanup_bucket

BUCKET = 'lab7-rs'
KEY = 'test/rs_verify.bin'
# Use a 64-byte payload — clean multiple of k=4 makes shard math easy to inspect
PAYLOAD = bytes(range(64))

s3 = get_s3_client()
ensure_bucket(s3, BUCKET)

# Upload
s3.put_object(Bucket=BUCKET, Key=KEY, Body=PAYLOAD)
print(f'✅ Uploaded {len(PAYLOAD)} bytes to s3://{BUCKET}/{KEY}')

# Verify our Python RS encoder would reconstruct the same data
our_shards = encode(PAYLOAD, K, M)
print(f'   Our RS({K},{M}) encoder produced {N} shards of {len(our_shards[0])} bytes')

# Simulate drive 1 failure via directory rename
CONTAINER = 'rustfs-server'
fail_drive(CONTAINER, BUCKET, drive=1)

# Read back from RustFS — it reconstructs transparently
body = s3.get_object(Bucket=BUCKET, Key=KEY)['Body'].read()
assert body == PAYLOAD, 'RustFS reconstruction failed!'
print(f'✅ RustFS reconstructed correctly from 5/6 shards (drive 1 offline)')

# Our decoder with shard 1 removed should produce the same result
available = {i: our_shards[i] for i in range(N) if i != 1}
our_recovered = decode(available, K, M, len(PAYLOAD))
assert our_recovered == PAYLOAD
print(f'✅ Our Python decoder also reconstructed correctly')
print(f'   Both produce: {our_recovered.hex()[:40]}...')

# Restore and cleanup
restore_drive(CONTAINER, BUCKET, drive=1)
cleanup_bucket(s3, BUCKET)

---
## 7 · Storage Efficiency vs. Fault Tolerance Trade-off

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

configs = [
    ('3× Replication',  3, 3, 2),   # k=3 copies, m=2 extra, 2 failures
    ('RS(4,2)',         4, 2, 2),   # this lab
    ('RS(4,4)',         4, 4, 4),
    ('RS(6,2)',         6, 2, 2),
    ('RS(10,4)',       10, 4, 4),
]

names = [c[0] for c in configs]
efficiencies = [c[1] / (c[1] + c[2]) * 100 for c in configs]
tolerances = [c[3] for c in configs]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Erasure Coding Schemes: Efficiency vs. Fault Tolerance', fontsize=13, fontweight='bold')

colors = ['#e74c3c' if n == 'RS(4,2)' else '#3498db' for n in names]
bars = ax1.barh(names, efficiencies, color=colors)
ax1.set_xlabel('Storage Efficiency (%)')
ax1.set_title('How much of stored bytes is actual data')
ax1.set_xlim(0, 100)
for bar, val in zip(bars, efficiencies):
    ax1.text(val + 1, bar.get_y() + bar.get_height() / 2, f'{val:.0f}%', va='center')

bars2 = ax2.barh(names, tolerances, color=colors)
ax2.set_xlabel('Drive Failures Tolerated')
ax2.set_title('Maximum concurrent drive failures survived')
for bar, val in zip(bars2, tolerances):
    ax2.text(val + 0.05, bar.get_y() + bar.get_height() / 2, str(val), va='center')

patch = mpatches.Patch(color='#e74c3c', label='This lab (RS(4,2))')
fig.legend(handles=[patch], loc='lower center', ncol=1)
plt.tight_layout(rect=[0, 0.05, 1, 1])
plt.show()

print('\nConclusion: RS(4,2) is a sweet spot — 67% efficiency with 2-drive tolerance.')
print('3× replication wastes 33% of storage for the same 2-failure tolerance.')

---
## Summary

You implemented **Reed-Solomon RS(4,2)** from scratch:

| Component | What it does |
|-----------|-------------|
| GF(2⁸) | Field arithmetic — XOR addition, log-table multiplication |
| Vandermonde matrix | Guarantees any k×k submatrix is invertible |
| Encoder | Splits data into k shards, computes m parity shards |
| Decoder | Inverts the submatrix of surviving shards, recovers original bytes |

The key insight: **any k=4 shards out of 6 are sufficient** because the Vandermonde
construction guarantees the corresponding submatrix is always invertible.

RustFS does the same math but with AVX2 SIMD acceleration — millions of bytes per second
instead of our pure-Python byte-at-a-time loop.
